# Retrieval Evaluation

Compare two retrieval approaches:
1. **Vector search** — pure semantic search with Qdrant
2. **Hybrid search** — vector + keyword boost re-ranking

Metrics:
- **Hit Rate** — was the correct document in the top-5 results?
- **MRR** (Mean Reciprocal Rank) — how high was the correct document ranked?

Prerequisites: run `evaluation-data-generation.ipynb` first.

In [ ]:
import os
import sys
import hashlib
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from qdrant_client import QdrantClient
from fastembed import TextEmbedding

# Add dataops_assistant to path for imports
sys.path.insert(0, "../dataops_assistant")

QDRANT_HOST = os.getenv("QDRANT_HOST", "localhost")
QDRANT_PORT = int(os.getenv("QDRANT_PORT", "6333"))
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "dataops_docs")
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
embedding_model = TextEmbedding(model_name=EMBEDDING_MODEL)

print(f"Connected to Qdrant at {QDRANT_HOST}:{QDRANT_PORT}")
print(f"Collection: {COLLECTION_NAME}")

In [ ]:
# Load ground truth
df_ground_truth = pd.read_csv("../data/ground-truth-retrieval.csv")
print(f"Ground truth pairs: {len(df_ground_truth)}")
df_ground_truth.head()

## Search functions

In [ ]:
def vector_search(query: str, num_results: int = 5) -> list:
    query_vector = list(embedding_model.embed([query]))[0].tolist()
    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        limit=num_results,
    )
    return [
        {
            "id": hashlib.md5(
                f"{hit.payload['filename']}_{hit.payload['chunk_index']}".encode()
            ).hexdigest()[:12],
            "score": hit.score,
            "text": hit.payload["text"],
        }
        for hit in results
    ]


def hybrid_search(query: str, num_results: int = 5) -> list:
    results = vector_search(query, num_results=num_results * 2)  # fetch more
    query_terms = set(query.lower().split())
    for r in results:
        text_terms = set(r["text"].lower().split())
        overlap = len(query_terms & text_terms)
        r["score"] += overlap * 0.01
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:num_results]


# Quick test
test_results = vector_search("how to define a dbt source")
print(f"Top result score: {test_results[0]['score']:.4f}")
print(f"Top result preview: {test_results[0]['text'][:100]}...")

## Evaluation metrics

In [ ]:
def hit_rate(relevance_scores: list) -> float:
    """Fraction of queries where correct doc appears in top-k."""
    return sum(any(row) for row in relevance_scores) / len(relevance_scores)


def mrr(relevance_scores: list) -> float:
    """Mean Reciprocal Rank."""
    scores = []
    for row in relevance_scores:
        for i, hit in enumerate(row):
            if hit:
                scores.append(1 / (i + 1))
                break
        else:
            scores.append(0)
    return sum(scores) / len(scores)


def evaluate_search(search_fn, ground_truth_df: pd.DataFrame) -> dict:
    relevance_scores = []
    for _, row in tqdm(ground_truth_df.iterrows(), total=len(ground_truth_df)):
        results = search_fn(row["question"])
        result_ids = [r["id"] for r in results]
        relevance = [doc_id == row["id"] for doc_id in result_ids]
        relevance_scores.append(relevance)
    return {
        "hit_rate": hit_rate(relevance_scores),
        "mrr": mrr(relevance_scores),
    }

## Run evaluation — Vector Search

In [ ]:
print("Evaluating vector search...")
vector_metrics = evaluate_search(vector_search, df_ground_truth)
print(f"Hit Rate: {vector_metrics['hit_rate']:.4f}")
print(f"MRR:      {vector_metrics['mrr']:.4f}")

## Run evaluation — Hybrid Search

In [ ]:
print("Evaluating hybrid search...")
hybrid_metrics = evaluate_search(hybrid_search, df_ground_truth)
print(f"Hit Rate: {hybrid_metrics['hit_rate']:.4f}")
print(f"MRR:      {hybrid_metrics['mrr']:.4f}")

## Results comparison

In [ ]:
results_df = pd.DataFrame([
    {"approach": "Vector Search", **vector_metrics},
    {"approach": "Hybrid Search", **hybrid_metrics},
])

results_df = results_df.set_index("approach")
results_df["hit_rate"] = results_df["hit_rate"].map("{:.4f}".format)
results_df["mrr"] = results_df["mrr"].map("{:.4f}".format)
results_df

In [ ]:
# Save results
results_df.to_csv("../data/retrieval-eval-results.csv")
print("Saved to ../data/retrieval-eval-results.csv")
print("\n→ Best approach for production: Hybrid Search" 
      if hybrid_metrics['hit_rate'] >= vector_metrics['hit_rate'] 
      else "\n→ Best approach for production: Vector Search")